In [28]:
%pip install pandas openpyxl pyarrow matplotlib seaborn scikit-learn statsmodels


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import pandas as pd

df = pd.read_excel("../data/loan_dataset_investor.xlsx")
df.to_parquet("../data/loans.parquet")   # use this from now on

In [30]:
df.shape

(789144, 22)

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 789144 entries, 0 to 789143
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       789144 non-null  str           
 1   country                       789144 non-null  str           
 2   loan_issued_at                789144 non-null  datetime64[us]
 3   early_repaid_at               229846 non-null  datetime64[us]
 4   issued_amount                 789144 non-null  float64       
 5   loan_status                   789144 non-null  str           
 6   initial_interest_rate         789144 non-null  float64       
 7   nr_of_payments                789144 non-null  int64         
 8   principal_balance             781431 non-null  float64       
 9   principal_debt                781431 non-null  float64       
 10  principal_paid_total          781431 non-null  float64       
 11  interest_paid_total     

In [32]:
print(df["is_default"].unique())

[ 0.  1. nan]


In [33]:
print(df["has_default_within_12_months"].unique())

[ 0.  1. nan]


In [34]:
print(df["customer_risk_rating"].unique())

<ArrowStringArray>
[nan, 'D', 'AA', 'B', 'A', 'C', 'F', 'G', 'E', 'HR']
Length: 10, dtype: str


In [35]:
df["is_default"] = df["is_default"].astype("Int8")
df["has_default_within_12_months"] = df["has_default_within_12_months"].astype("Int8")

for col in ["country", "loan_status", "loan_status_risk", "customer_risk_rating"]:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 789144 entries, 0 to 789143
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       789144 non-null  str           
 1   country                       789144 non-null  category      
 2   loan_issued_at                789144 non-null  datetime64[us]
 3   early_repaid_at               229846 non-null  datetime64[us]
 4   issued_amount                 789144 non-null  float64       
 5   loan_status                   789144 non-null  category      
 6   initial_interest_rate         789144 non-null  float64       
 7   nr_of_payments                789144 non-null  int64         
 8   principal_balance             781431 non-null  float64       
 9   principal_debt                781431 non-null  float64       
 10  principal_paid_total          781431 non-null  float64       
 11  interest_paid_total     

In [46]:
order = ["AA", "A", "B", "C", "D", "E", "F", "G", "HR"]  # replace with your actual values
df["customer_risk_rating"] = pd.Categorical(df["customer_risk_rating"], categories=order, ordered=True)

In [37]:
df = df.drop(columns=["nr_of_payments"])

In [38]:
df["is_default"].isna().sum()

np.int64(7713)

In [39]:
df["combined_income"].isna().sum()

np.int64(273160)

In [47]:
df["customer_risk_rating"].isna().sum()


np.int64(338509)

In [41]:
df["has_default_within_12_months"].isna().sum()

np.int64(182280)

In [42]:
df["months_in_default"].isna().sum()

np.int64(681608)

**The total dataset is (789144), but when we check here at months_in_defalt, it shows that missing values are (681608). But none is not default. Blank means “not in default” rather than “unknown”. As per the dataset dictionary, the number of months since default occurred, NULL if not default. Amount of months in default.**

In [44]:
df.shape

(789144, 21)

In [45]:
df.describe()

,loan_issued_at,early_repaid_at,issued_amount,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,interest_paid_total,is_default,next_payment_date_local,months_in_default,months_on_book,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months
count,789144,229846,789144.000000,789144.000000,781431.000000,781431.000000,781431.000000,781431.000000,781431.0,329989,107536.000000,779947.000000,781431.000000,789144.000000,515984.000000,606864.0
mean,2023-03-13 13:55:54.384715,2023-08-10 23:19:23.015236,2231.619091,0.215558,1005.995033,268.415089,1084.738436,481.136600,0.137614,2026-04-30 05:36:14.810675,33.774810,15.909749,1648.305292,56.835717,2535.931052,0.14273
min,2009-02-28 18:05:13,2009-05-21 00:00:00,6.390000,0.010000,-34.200000,0.000000,0.000000,0.000000,0.0,2009-04-15 00:00:00,0.000000,0.000000,0.000000,1.000000,0.000000,0.0
25%,2021-09-21 00:29:36.500000,2022-05-18 00:00:00,530.000000,0.174800,0.000000,0.000000,94.600000,32.755000,0.0,2026-07-31 00:00:00,8.000000,5.000000,213.100000,36.000000,1681.265000,0.0
50%,2024-05-03 10:42:36.385000,2024-08-14 00:00:00,1349.000000,0.187700,117.310000,0.000000,448.900000,153.430000,0.0,2026-08-05 00:00:00,20.000000,11.000000,718.600000,60.000000,2329.800000,0.0
75%,2025-06-23 13:51:50.849500,2025-09-15 00:00:00,3627.000000,0.221800,1135.975615,0.000000,1359.615000,548.015000,0.0,2026-08-17 00:00:00,50.000000,22.000000,2240.250000,60.000000,3024.130000,0.0
max,2026-06-30 02:39:06.073000,2026-07-20 00:00:00,30000.000000,2.643100,30000.000000,19744.280000,30000.000000,32975.500000,1.0,2027-06-30 00:00:00,199.000000,161.000000,40524.500000,180.000000,177871.700000,1.0
std,NaN,NaN,2384.928361,0.134228,1937.818672,967.052316,1635.485612,905.037273,0.344495,NaN,32.654157,16.197289,2279.612180,29.099602,1555.381021,0.349798


In [ ]:
df.isna().mean().sort_values(ascending=False)   # missing values

In [20]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,681608,86.4
early_repaid_at,559298,70.9
next_payment_date_local,459155,58.2
customer_risk_rating,338509,42.9
combined_income,273160,34.6
has_default_within_12_months,182280,23.1
months_on_book,9197,1.2
principal_balance,7713,1.0
principal_debt,7713,1.0
principal_paid_total,7713,1.0


In [53]:
df[df["is_default"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,is_default,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating
242,0202E9E8-7DEC-492B-8E99-A4750116A711,Estonia,2015-04-10 08:10:00,2015-04-10,2000.0,Returned,0.2159,NaN,NaN,NaN,...,<NA>,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
297,02F69B8A-0EED-4AA9-BAB4-A41601550BE6,Spain,2015-01-13 20:20:01,2015-02-09,500.0,Repaid,0.7373,NaN,NaN,NaN,...,<NA>,NaT,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN
348,040457D8-32F7-447E-8185-A46900C06742,Estonia,2015-03-29 11:10:39,2015-03-30,1000.0,Returned,0.1769,NaN,NaN,NaN,...,<NA>,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
436,05A09DD4-EB44-40D5-9AF7-A48A012B84CD,Estonia,2015-05-06 18:28:03,2015-05-07,6000.0,Returned,0.2739,NaN,NaN,NaN,...,<NA>,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
447,05DA1697-9E05-4EE4-8F73-A452015B688B,Spain,2015-03-06 21:00:03,2015-04-02,600.0,Repaid,0.2470,NaN,NaN,NaN,...,<NA>,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788995,FCB8E77B-D235-4DAD-84D1-A4820119C184,Finland,2015-05-05 12:10:21,2015-05-05,4000.0,Returned,0.2410,NaN,NaN,NaN,...,<NA>,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
789008,FD126AD8-4E82-4830-B6C7-A424008CB02F,Spain,2015-01-20 18:25:09,2015-02-02,1000.0,Repaid,0.3112,NaN,NaN,NaN,...,<NA>,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN
789009,FD1333EF-1432-467F-9BED-A36A00A518A4,Estonia,2014-07-16 11:10:00,2014-07-24,500.0,Repaid,0.3400,NaN,NaN,NaN,...,<NA>,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN
789096,FEF91A4C-C485-405D-9263-A3920157F836,Spain,2014-09-05 20:05:05,2014-09-25,1055.0,Repaid,0.3200,NaN,NaN,NaN,...,<NA>,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN


In [55]:
df["default_flag"] = df["is_default"].fillna(0).astype(int)

In [57]:
print(df["is_default"].value_counts(dropna=False))
print(df["default_flag"].value_counts())

is_default
0       673895
1       107536
<NA>      7713
Name: count, dtype: Int64
default_flag
0    681608
1    107536
Name: count, dtype: int64


In [58]:
df[df["combined_income"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
0,0000B9FA-5598-4130-9143-AB63012DC8D8,Estonia,2020-02-16 20:29:49,2021-02-15,6804.00,Repaid,0.3982,0.00,0.00,6804.00,...,NaT,NaN,12.0,Paid Up,8699.58,60,NaN,0,NaN,0
1,0001463B-E6E2-4A6D-915E-AD570117C04D,Estonia,2021-06-30 18:34:44,2022-01-03,531.00,Repaid,0.1879,0.00,0.00,531.00,...,NaT,NaN,6.0,Paid Up,588.94,36,NaN,0,NaN,0
2,001B7346-9068-4A70-98E3-AE0400909BB5,Finland,2021-12-17 14:41:57,2023-02-03,498.48,Repaid,0.1990,0.00,0.00,498.48,...,NaT,NaN,13.0,Paid Up,605.25,24,NaN,0,NaN,0
4,0022E68D-A2EC-4CBF-B8B4-AD090117350B,Estonia,2021-04-13 17:07:35,NaT,531.00,Defaulted,0.3930,360.46,360.46,170.54,...,NaT,24.0,39.0,Defaulted,736.39,60,NaN,0,NaN,1
7,003E4687-1045-4B78-BD2F-AAB10076BD96,Estonia,2019-08-22 09:25:25,NaT,9675.00,Repaid,0.2338,0.00,0.00,2644.17,...,NaT,NaN,26.0,Written Off,6639.74,60,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789131,FFB56D26-7F37-41CB-9282-A9C400FD8077,Finland,2018-12-28 15:25:06,2024-06-11,2125.00,Repaid,0.5787,0.00,0.00,2125.00,...,NaT,NaN,66.0,Paid Up,8889.10,60,NaN,0,NaN,0
789139,FFE27BDC-CB73-4D0E-BBB0-A30800F274A8,Spain,2014-04-23 14:31:09,NaT,10000.00,Defaulted,0.2500,7972.45,7972.45,2027.55,...,NaT,121.0,26.0,Defaulted,6861.35,60,NaN,0,NaN,1
789140,FFE3C330-C6F6-4041-982E-ABA500E5F64A,Estonia,2020-04-22 14:43:42,NaT,1052.00,Repaid,0.3786,0.00,0.00,779.07,...,NaT,NaN,5.0,Written Off,831.70,60,NaN,1,NaN,0
789141,FFEE9D4E-255A-4894-A0D0-ADC00090368C,Estonia,2021-10-13 09:59:02,NaT,2764.00,Defaulted,0.3600,2283.97,2283.97,480.03,...,NaT,45.0,12.0,Defaulted,1204.10,60,NaN,0,NaN,1


In [59]:
df[df["customer_risk_rating"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
0,0000B9FA-5598-4130-9143-AB63012DC8D8,Estonia,2020-02-16 20:29:49.000,2021-02-15,6804.00,Repaid,0.3982,0.00,0.00,6804.00,...,NaT,NaN,12.0,Paid Up,8699.58,60,NaN,0,NaN,0
1,0001463B-E6E2-4A6D-915E-AD570117C04D,Estonia,2021-06-30 18:34:44.000,2022-01-03,531.00,Repaid,0.1879,0.00,0.00,531.00,...,NaT,NaN,6.0,Paid Up,588.94,36,NaN,0,NaN,0
2,001B7346-9068-4A70-98E3-AE0400909BB5,Finland,2021-12-17 14:41:57.000,2023-02-03,498.48,Repaid,0.1990,0.00,0.00,498.48,...,NaT,NaN,13.0,Paid Up,605.25,24,NaN,0,NaN,0
4,0022E68D-A2EC-4CBF-B8B4-AD090117350B,Estonia,2021-04-13 17:07:35.000,NaT,531.00,Defaulted,0.3930,360.46,360.46,170.54,...,NaT,24.0,39.0,Defaulted,736.39,60,NaN,0,NaN,1
7,003E4687-1045-4B78-BD2F-AAB10076BD96,Estonia,2019-08-22 09:25:25.000,NaT,9675.00,Repaid,0.2338,0.00,0.00,2644.17,...,NaT,NaN,26.0,Written Off,6639.74,60,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789139,FFE27BDC-CB73-4D0E-BBB0-A30800F274A8,Spain,2014-04-23 14:31:09.000,NaT,10000.00,Defaulted,0.2500,7972.45,7972.45,2027.55,...,NaT,121.0,26.0,Defaulted,6861.35,60,NaN,0,NaN,1
789140,FFE3C330-C6F6-4041-982E-ABA500E5F64A,Estonia,2020-04-22 14:43:42.000,NaT,1052.00,Repaid,0.3786,0.00,0.00,779.07,...,NaT,NaN,5.0,Written Off,831.70,60,NaN,1,NaN,0
789141,FFEE9D4E-255A-4894-A0D0-ADC00090368C,Estonia,2021-10-13 09:59:02.000,NaT,2764.00,Defaulted,0.3600,2283.97,2283.97,480.03,...,NaT,45.0,12.0,Defaulted,1204.10,60,NaN,0,NaN,1
789142,FFF26F90-95F8-4410-855F-AC7900E7A5E2,Estonia,2020-11-20 14:31:21.000,NaT,531.00,Repaid,0.2605,0.00,0.00,444.90,...,NaT,NaN,10.0,Written Off,497.87,12,NaN,1,NaN,0


In [61]:
df[df["has_default_within_12_months"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
11,004BA4A6-006A-4BCC-B352-B3DE00621512,Netherlands,2026-01-26 09:59:34.673,NaT,1976.0,Active,0.0714,1722.95,0.00,253.05,...,2026-08-11,NaN,5.0,Active,322.67,42,3207.71,<NA>,D,0
13,0057A33D-DFCB-4B6B-A7C1-B35F00D6DEAD,Finland,2025-09-21 13:07:20.913,2026-02-06,400.0,Repaid,0.1748,0.00,0.00,400.00,...,NaT,NaN,4.0,Paid Up,431.20,60,1775.29,<NA>,A,0
27,00A105D1-754B-465A-94DC-B3BB011FF83F,Finland,2025-12-22 17:38:40.053,NaT,6100.0,Active,0.1748,6100.00,0.00,0.00,...,2026-08-19,NaN,7.0,Active,699.10,90,3129.87,<NA>,AA,0
33,00C0588D-1402-494B-A1B7-B38500EC428E,Finland,2025-10-29 14:51:57.547,NaT,1000.0,Active,0.1499,965.58,0.00,34.42,...,2026-08-17,NaN,8.0,Active,166.30,120,3083.31,<NA>,AA,0
52,01275033-46F5-4705-8BF4-B38A013D27E8,Finland,2025-11-03 19:24:30.127,NaT,400.0,Active,0.1748,389.50,1.40,10.50,...,2026-08-18,NaN,8.0,Active,62.58,120,2393.95,<NA>,A,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789113,FF6F39B8-3CAA-4571-A216-B32A016962CE,Netherlands,2025-08-02 17:29:35.877,NaT,3120.0,Active,0.0720,1745.47,0.00,1374.53,...,2026-08-03,NaN,11.0,Active,1566.52,24,3604.15,<NA>,AA,0
789114,FF73C7F4-507B-4873-8BA5-B3810016EEE7,Finland,2025-10-25 07:19:24.050,NaT,200.0,Active,0.1748,125.20,0.00,74.80,...,2026-08-10,NaN,8.0,Active,99.13,12,3149.72,<NA>,B,0
789118,FF7D5C4B-95F6-4287-944B-B33A010C61C2,Estonia,2025-08-15 16:35:24.667,NaT,2658.0,Active,0.1999,2481.75,0.00,176.25,...,2026-08-10,NaN,11.0,Active,734.17,84,1900.00,<NA>,B,0
789133,FFC4C6D3-AF50-4633-BE22-B39D0167A498,Estonia,2025-11-22 21:54:35.463,NaT,106.0,Active,0.1975,64.34,0.00,41.66,...,2026-08-10,NaN,8.0,Active,54.15,12,2100.00,<NA>,AA,0


**They repaid even before one year, but in repaid_amount_total, we have as missing values. So can we delete this?**

In [64]:
df[df["repaid_amount_total"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
242,0202E9E8-7DEC-492B-8E99-A4750116A711,Estonia,2015-04-10 08:10:00,2015-04-10,2000.0,Returned,0.2159,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
297,02F69B8A-0EED-4AA9-BAB4-A41601550BE6,Spain,2015-01-13 20:20:01,2015-02-09,500.0,Repaid,0.7373,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN,0
348,040457D8-32F7-447E-8185-A46900C06742,Estonia,2015-03-29 11:10:39,2015-03-30,1000.0,Returned,0.1769,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
436,05A09DD4-EB44-40D5-9AF7-A48A012B84CD,Estonia,2015-05-06 18:28:03,2015-05-07,6000.0,Returned,0.2739,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
447,05DA1697-9E05-4EE4-8F73-A452015B688B,Spain,2015-03-06 21:00:03,2015-04-02,600.0,Repaid,0.2470,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788995,FCB8E77B-D235-4DAD-84D1-A4820119C184,Finland,2015-05-05 12:10:21,2015-05-05,4000.0,Returned,0.2410,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
789008,FD126AD8-4E82-4830-B6C7-A424008CB02F,Spain,2015-01-20 18:25:09,2015-02-02,1000.0,Repaid,0.3112,NaN,NaN,NaN,...,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN,0
789009,FD1333EF-1432-467F-9BED-A36A00A518A4,Estonia,2014-07-16 11:10:00,2014-07-24,500.0,Repaid,0.3400,NaN,NaN,NaN,...,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN,0
789096,FEF91A4C-C485-405D-9263-A3920157F836,Spain,2014-09-05 20:05:05,2014-09-25,1055.0,Repaid,0.3200,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0


**For the first and second person, they pay back within one month, so there is no money to pay back and principal_balance is the balance of money to pay back so this null values is meaningless, we should remove.**

In [66]:
df[df["principal_balance"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
242,0202E9E8-7DEC-492B-8E99-A4750116A711,Estonia,2015-04-10 08:10:00,2015-04-10,2000.0,Returned,0.2159,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
297,02F69B8A-0EED-4AA9-BAB4-A41601550BE6,Spain,2015-01-13 20:20:01,2015-02-09,500.0,Repaid,0.7373,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN,0
348,040457D8-32F7-447E-8185-A46900C06742,Estonia,2015-03-29 11:10:39,2015-03-30,1000.0,Returned,0.1769,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
436,05A09DD4-EB44-40D5-9AF7-A48A012B84CD,Estonia,2015-05-06 18:28:03,2015-05-07,6000.0,Returned,0.2739,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
447,05DA1697-9E05-4EE4-8F73-A452015B688B,Spain,2015-03-06 21:00:03,2015-04-02,600.0,Repaid,0.2470,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788995,FCB8E77B-D235-4DAD-84D1-A4820119C184,Finland,2015-05-05 12:10:21,2015-05-05,4000.0,Returned,0.2410,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
789008,FD126AD8-4E82-4830-B6C7-A424008CB02F,Spain,2015-01-20 18:25:09,2015-02-02,1000.0,Repaid,0.3112,NaN,NaN,NaN,...,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN,0
789009,FD1333EF-1432-467F-9BED-A36A00A518A4,Estonia,2014-07-16 11:10:00,2014-07-24,500.0,Repaid,0.3400,NaN,NaN,NaN,...,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN,0
789096,FEF91A4C-C485-405D-9263-A3920157F836,Spain,2014-09-05 20:05:05,2014-09-25,1055.0,Repaid,0.3200,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0


**For all of these people, they returned the money. So we do not have principal_debt as numbers in this column, which make sense not to have it since they pay back. But for principal_balance and principal_paid_total, we should have some number that they repay. Here we do not have it, so this is really missing values. Sould we delect this or how can we handle this?**

In [67]:
df[df["principal_debt"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
242,0202E9E8-7DEC-492B-8E99-A4750116A711,Estonia,2015-04-10 08:10:00,2015-04-10,2000.0,Returned,0.2159,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
297,02F69B8A-0EED-4AA9-BAB4-A41601550BE6,Spain,2015-01-13 20:20:01,2015-02-09,500.0,Repaid,0.7373,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN,0
348,040457D8-32F7-447E-8185-A46900C06742,Estonia,2015-03-29 11:10:39,2015-03-30,1000.0,Returned,0.1769,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
436,05A09DD4-EB44-40D5-9AF7-A48A012B84CD,Estonia,2015-05-06 18:28:03,2015-05-07,6000.0,Returned,0.2739,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
447,05DA1697-9E05-4EE4-8F73-A452015B688B,Spain,2015-03-06 21:00:03,2015-04-02,600.0,Repaid,0.2470,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788995,FCB8E77B-D235-4DAD-84D1-A4820119C184,Finland,2015-05-05 12:10:21,2015-05-05,4000.0,Returned,0.2410,NaN,NaN,NaN,...,NaT,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
789008,FD126AD8-4E82-4830-B6C7-A424008CB02F,Spain,2015-01-20 18:25:09,2015-02-02,1000.0,Repaid,0.3112,NaN,NaN,NaN,...,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN,0
789009,FD1333EF-1432-467F-9BED-A36A00A518A4,Estonia,2014-07-16 11:10:00,2014-07-24,500.0,Repaid,0.3400,NaN,NaN,NaN,...,NaT,NaN,0.0,Cancelled,NaN,36,NaN,0,NaN,0
789096,FEF91A4C-C485-405D-9263-A3920157F836,Spain,2014-09-05 20:05:05,2014-09-25,1055.0,Repaid,0.3200,NaN,NaN,NaN,...,NaT,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
